# Day 7: Initial Dataset Inspection (OCC Videos)
This notebook inspects the raw OCC video sequences without modifying or cleaning the data. It serves to validate the metadata, labels, and coverage.

In [ ]:
import cv2
import os
import pandas as pd
import matplotlib.pyplot as plt
import glob

# Define raw data paths
RAW_VIDEO_DIR = './data/raw/videos/'
GROUND_TRUTH_PATH = './data/raw/ground_truth.csv'
print('Data paths configured successfully.')

## 1. Inspect Ground Truth Labels & Targets

In [ ]:
# Load ground truth mapping
try:
    df_labels = pd.read_csv(GROUND_TRUTH_PATH)
    print("Total Sequences:", len(df_labels))
    print("\nMissing Values:\n", df_labels.isnull().sum())
    print("\nFirst 5 entries:\n", df_labels.head())
except FileNotFoundError:
    print("[WARN] Ground truth file not found. Ensure it is placed in:", GROUND_TRUTH_PATH)


## 2. Inspect Video Files and Extract Metadata

In [ ]:
video_files = glob.glob(os.path.join(RAW_VIDEO_DIR, '*.mp4')) + glob.glob(os.path.join(RAW_VIDEO_DIR, '*.avi'))
print(f"Found {len(video_files)} raw video files.")

metadata_list = []
for vid_path in video_files:
    cap = cv2.VideoCapture(vid_path)
    if cap.isOpened():
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration = frame_count / fps if fps > 0 else 0
        
        metadata_list.append({
            'filename': os.path.basename(vid_path),
            'fps': fps,
            'total_frames': frame_count,
            'resolution': f"{width}x{height}",
            'duration_sec': round(duration, 2)
        })
        cap.release()

df_metadata = pd.DataFrame(metadata_list)
if not df_metadata.empty:
    display(df_metadata.head())
else:
    print("No video metadata extracted. Check directory.")
